# AutoMarket AI: Cloud-Native Valuation Pipeline
**Author:** Abayomi Nelson  
**Objective:** Ingest partitioned automotive market data, perform exploratory data analysis, and train a categorical XGBoost model to predict fair market vehicle values. The final artifacts are exported for deployment in a Streamlit web application.

## Code (Imports & Environment Setup)

In [14]:
import pandas as pd
import os
import json
import joblib
import warnings
from pathlib import Path
from xgboost import XGBRegressor

# Silence Pandas FutureWarnings for clean notebook presentation
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('display.max_columns', None) # Show all columns during EDA

## Code (Data Ingestion & Assembly)

In [15]:
print("Gathering partitioned CSV files...")

# Use the dynamic cache path
dataset_path = "/Users/abayominelson/.cache/kagglehub/datasets/rebrowser/carfax-dataset/versions/40"
all_csv_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith('.csv'):
            all_csv_files.append(os.path.join(root, file))

# Load and stitch files, dropping completely empty columns
df_list = [pd.read_csv(file, low_memory=False).dropna(axis=1, how='all') for file in all_csv_files]
df = pd.concat(df_list, ignore_index=True)

print(f"Success! Loaded {len(df):,} total records.")

Gathering partitioned CSV files...
Success! Loaded 24,047 total records.


## Part 1: Exploratory Data Analysis (EDA)
Before feature engineering, we analyze the raw dataset to understand market distribution, dealer behavior, and premium vehicle traits (e.g., 1-owner histories, no accidents).

## Code (Publisher's EDA)

In [16]:
print("--- Top 10 Makes ---")
print(df['make'].value_counts().head(10).to_string())
print("\n--- Average Mileage by Body Style ---")
print(df.groupby('bodyStyle')['mileage'].mean().sort_values().to_string())

# Safe check for specific Carfax columns before running deeper EDA
if 'noAccidents' in df.columns and 'sellerState' in df.columns:
    print("\n--- Top 10 States for No-Accident Listings ---")
    clean = df[df['noAccidents'] == True]
    print(clean['sellerState'].value_counts().head(10).to_string())

if 'sellerReviewCount' in df.columns and 'sellerRating' in df.columns:
    print("\n--- Top Rated Dealers (50+ Reviews) ---")
    top_dealers = df[df['sellerReviewCount'] >= 50]
    print(top_dealers.groupby('sellerId')['sellerRating']
          .first().sort_values(ascending=False).head(10).to_string())

if 'drivetrain' in df.columns and 'badge' in df.columns:
    print("\n--- Deal Badges by Drivetrain ---")
    print(pd.crosstab(df['drivetrain'], df['badge']).to_string())

if all(col in df.columns for col in ['oneOwner', 'bodyStyle', 'serviceRecords']):
    print("\n--- Premium Filter: One-Owner SUVs with Service Records ---")
    one_owner_suvs = df[
        (df['oneOwner'] == True) &
        (df['bodyStyle'] == 'SUV') &
        (df['serviceRecords'] == True)
    ]
    print(f"Total Found: {len(one_owner_suvs)}")
    print(one_owner_suvs[['make', 'model', 'year', 'mileage', 'sellerState']]
          .sort_values('mileage').head(10).to_string(index=False))

--- Top 10 Makes ---
make
Ford             3805
Chevrolet        2766
Toyota           2203
Ram              1648
Honda            1409
GMC              1385
Mercedes-Benz    1160
BMW              1143
Nissan            851
Jeep              647

--- Average Mileage by Body Style ---
bodyStyle
Unspecified      542.000000
Coupe          29688.836576
Convertible    42613.694805
SUV            49327.289443
Wagon          53742.716946
Sedan          54935.977210
Pickup         59113.335306
Minivan        64876.427614
Van            69549.047118
Chassis        73426.100917
Hatchback      79914.593812

--- Top 10 States for No-Accident Listings ---
sellerState
TX    1840
FL    1591
CA    1568
NC     715
OH     653
NJ     608
NY     595
GA     590
PA     582
AZ     533

--- Top Rated Dealers (50+ Reviews) ---
sellerId
BURJATM001    5.0
DW37GT918V    5.0
GLDJSMT6DV    5.0
QWXWSU6VG1    5.0
BBVM8M67U1    5.0
VSCSSP8001    5.0
QRNQGA1001    5.0
2TED3IP7UL    5.0
89QAU3N001    5.0
4T4YT4F001    5

## Part 2: Data Cleaning & Feature Engineering
Preparing the dataset for the XGBoost Regressor. We will isolate the predictive features, handle missing values, and engineer a continuous `vehicle_age` feature to replace the static `year` column.

In [35]:
print("Executing Data Cleaning & Feature Engineering...")

features = ['make', 'model', 'bodyStyle', 'mileage', 'year', 'onePrice']

# 1. Slice the dataframe
clean_df = df[features].copy()
print(f"1. Starting with {len(clean_df):,} raw records.")

# 2. Force variables to be strict numbers
clean_df['onePrice'] = pd.to_numeric(clean_df['onePrice'], errors='coerce')
clean_df['mileage'] = pd.to_numeric(clean_df['mileage'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')

# 3. Drop missing values
clean_df = clean_df.dropna()
print(f"2. After dropping nulls, we have {len(clean_df):,} valid records left.")

# 4. Engineer Age feature
clean_df['vehicle_age'] = 2026 - clean_df['year']
clean_df = clean_df.drop(columns=['year'])

# 5. Rename 'onePrice' to 'price' so downstream code works seamlessly
clean_df = clean_df.rename(columns={'onePrice': 'price_col'})

print(f"3. Cleaned dataset ready with {len(clean_df):,} pure numerical records.")
clean_df.head()

Executing Data Cleaning & Feature Engineering...
1. Starting with 24,047 raw records.
2. After dropping nulls, we have 20,701 valid records left.
3. Cleaned dataset ready with 20,701 pure numerical records.


,make,model,bodyStyle,mileage,price_col,vehicle_age
0,Honda,Pilot,SUV,23325.0,41240.0,1
1,Chevrolet,Silverado 1500,Pickup,85000.0,18520.0,6
4,Kia,Optima,Sedan,57226.0,14330.0,6
5,Nissan,Versa,Sedan,120315.0,9250.0,6
6,Honda,Insight,Sedan,121720.0,14620.0,6


## Part 3: Model Preparation & Artifact Generation
To ensure the Streamlit web application runs efficiently, we generate a JSON mapping of all available Make/Model combinations. This prevents the web app from needing to load the raw dataframe into memory.

## Code (UI Mapping & Categorical Encoding)

In [36]:
print("Generating Deep UI mapping dictionary (Make > Model > BodyStyles)...")

# Create a nested dictionary structure
# Example: {"Acura": {"TL": ["Sedan"], "MDX": ["SUV"]}}
deep_mapping = {}
for make in clean_df['make'].unique():
    make_df = clean_df[clean_df['make'] == make]
    deep_mapping[make] = make_df.groupby('model')['bodyStyle'].unique().apply(list).to_dict()

with open('make_model_mapping.json', 'w') as f:
    json.dump(deep_mapping, f)
    
print("Success: Deep mapping saved to make_model_mapping.json")

Generating Deep UI mapping dictionary (Make > Model > BodyStyles)...
Success: Deep mapping saved to make_model_mapping.json


## Part 4: XGBoost Model Training
Training the model using XGBoost's native categorical support (`enable_categorical=True`), which handles high-cardinality text data (like hundreds of car models) without requiring memory-intensive One-Hot Encoding.

## Code (Model Training & Export)

In [38]:
# XGBoost enable_categorical=True ONLY works if the dtypes are 'category'
categorical_cols = ['make', 'model', 'bodyStyle']
for col in categorical_cols:
    clean_df[col] = clean_df[col].astype('category')

# 2. Train/Test Split logic
X = clean_df.drop(columns=['price_col'])
y = clean_df['price_col']

print(f"Training XGBoost Model on {len(clean_df):,} validated records...")

# 3. Initialize and fit the model
model = XGBRegressor(
    enable_categorical=True, 
    tree_method='hist', 
    n_estimators=100,
    random_state=42
)
model.fit(X, y)

# 4. Export the trained model weights
joblib.dump(model, 'universal_market_predictor.pkl')

print("🚀 Pipeline Complete! Model (.pkl) successfully saved.")

Training XGBoost Model on 20,701 validated records...
🚀 Pipeline Complete! Model (.pkl) successfully saved.


## Validation

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Split data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Re-fit on training data and predict on test data
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 3. Calculate Performance Metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"--- Model Performance Metrics ---")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"R-Squared Score: {r2:.4f}")

# 4. Quick Sanity Check: Compare actual vs. predicted for first 5 rows
comparison = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}).head(5)
print("\n--- Actual vs. Predicted (Sample) ---")
print(comparison)

--- Model Performance Metrics ---
Mean Absolute Error (MAE): $4,722.82
R-Squared Score: 0.8295

--- Actual vs. Predicted (Sample) ---
        Actual     Predicted
10546  55590.0  46738.558594
23553  15620.0  16608.640625
19828  19930.0  25074.591797
13584  33520.0  34265.875000
4899   66650.0  71307.109375
